In [18]:
import pandas as pd
import numpy as np
import shap
import joblib
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

ind_hosp = pd.read_parquet('ind_hosp.parquet')
calibrated = joblib.load('lightgbm.pkl')
base_model = calibrated.calibrated_classifiers_[0].estimator

cols_to_drop = ['subject_id', 'hadm_id', 'dischtime', 'current_date', 
                'target_readmission_30d', 'los']
cols_to_drop = [c for c in cols_to_drop if c in ind_hosp.columns]
X = ind_hosp.drop(columns=cols_to_drop)

bool_cols = X.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X[bool_cols] = X[bool_cols].astype(int)

patient_target = ind_hosp.groupby('subject_id')['target_readmission_30d'].max().reset_index()
patient_target.columns = ['subject_id', 'has_readmission']

train_val_ids, test_ids = train_test_split(
    patient_target['subject_id'],
    test_size=0.2,
    random_state=42,
    stratify=patient_target['has_readmission']
)

test_mask = ind_hosp['subject_id'].isin(test_ids)
X_test = X[test_mask].copy()
y_test = ind_hosp.loc[test_mask, 'target_readmission_30d'].copy()

sample_size = len(X_test)
X_sample = X_test.sample(sample_size, random_state=42)
y_sample = y_test.loc[X_sample.index]

In [19]:
from tqdm import tqdm
print('Calculating SHAP...')
explainer = shap.TreeExplainer(base_model)
batch_size = 1000
n_batches = int(np.ceil(len(X_sample) / batch_size))

shap_values_list = []

for i in tqdm(range(n_batches), desc="Computing SHAP"):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(X_sample))
    X_batch = X_sample.iloc[start_idx:end_idx]
    
    shap_batch = explainer.shap_values(X_batch)
    shap_values_list.append(shap_batch)

shap_values = np.vstack(shap_values_list)
print(f"SHAP values shape: {shap_values.shape}")
print('Done')

Calculating SHAP...


Computing SHAP: 100%|██████████| 366/366 [11:00<00:00,  1.80s/it]


SHAP values shape: (365442, 132)
Done


In [24]:
shap_mean_original = shap_values.mean(axis=0)
shap_mean_abs = np.abs(shap_values).mean(axis=0)

shap_df = pd.DataFrame({
    'feature': X_sample.columns,
    'shap_mean': shap_mean_original,
    'shap_abs_mean': shap_mean_abs,
    'shap_std': shap_values.std(axis=0),
    'shap_min': shap_values.min(axis=0),
    'shap_max': shap_values.max(axis=0),
})

icd_cols = [col for col in X.columns if col.startswith('icd_')]
ccsr_cols = [col for col in X.columns if col.startswith('ccsr_')]
lab_cols = [col for col in X.columns if col.startswith('lab_') and col.endswith('_daily')]

features_to_analyze = (
    icd_cols +
    ccsr_cols +
    lab_cols +
    [
        'num_diagnoses',
        'num_chronic',
        'comorbidity_score',
        'num_medications_daily',
        'prior_admissions_12mo',
        'cumulative_procedures',
        'cumulative_medications',
        'num_procedures_daily',
        'gender_male',
        'age'
    ]
)

shap_df_filtered = shap_df[shap_df['feature'].isin(features_to_analyze)]
shap_df_filtered.to_csv('shap.csv', index=False)

In [8]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

deltas_df = pd.read_csv('delta_summary.csv')
shap_df = pd.read_csv('shap.csv')

comparison = shap_df.merge(
    deltas_df[['feature', 'delta_mean', 'delta_std']],
    on='feature',
    how='inner'
)

In [9]:
spearman_mean = spearmanr(comparison['delta_mean'], comparison['shap_mean'])[0]
spearman_abs = spearmanr(comparison['delta_mean'].abs(), comparison['shap_abs_mean'])[0]

print(f"Spearman (Δ vs SHAP_mean):    {spearman_mean:.4f}")
print(f"Spearman (|Δ| vs SHAP_abs):   {spearman_abs:.4f}")

comparison['delta_dir'] = np.sign(comparison['delta_mean'])
comparison['shap_dir'] = np.sign(comparison['shap_mean'])
comparison['direction_agreement'] = comparison['delta_dir'] == comparison['shap_dir']
agreement_rate = comparison['direction_agreement'].mean()
print(f"Sign direction agreement rate: {agreement_rate:.2%}")

Spearman (Δ vs SHAP_mean):    -0.1752
Spearman (|Δ| vs SHAP_abs):   0.7769
Sign direction agreement rate: 48.33%


In [12]:
comparison['delta_rank'] = comparison['delta_mean'].abs().rank(ascending=False)
comparison['shap_rank'] = comparison['shap_mean'].abs().rank(ascending=False)

results_df = pd.DataFrame({
    'feature': comparison['feature'],
    'delta': comparison['delta_mean'],
    'shap': comparison['shap_mean'],
    'delta_rank': comparison['delta_rank'],
    'shap_rank': comparison['shap_rank'],
    'rank_diff': comparison['delta_rank'] - comparison['shap_rank'],
})

results_df['delta_dir'] = results_df['delta'].apply(lambda x: '↑' if x > 0 else '↓')
results_df['shap_dir'] = results_df['shap'].apply(lambda x: '↑' if x > 0 else '↓')
results_df['dir_match'] = results_df['delta_dir'] == results_df['shap_dir']

results_df = results_df.sort_values('rank_diff', ascending=False)

In [16]:
top_delta_positive = results_df.nlargest(10, 'delta')
top_delta_negative = results_df.nsmallest(10, 'delta')

print(top_delta_positive[['feature', 'delta', 'shap', 'delta_rank', 'shap_rank']].to_string(index=False))
print()
print(top_delta_negative[['feature', 'delta', 'shap', 'delta_rank', 'shap_rank']].to_string(index=False))

              feature  delta      shap  delta_rank  shap_rank
      lab_50983_daily 0.0256  0.000144         3.0       44.0
    comorbidity_score 0.0203 -0.010927         4.0        3.0
prior_admissions_12mo 0.0193 -0.014342         5.0        1.0
      lab_51248_daily 0.0155 -0.000550         8.0       22.0
      lab_51221_daily 0.0135  0.000070        10.0       49.0
      lab_50882_daily 0.0109 -0.000136        13.0       45.0
      lab_51265_daily 0.0084 -0.001395        16.0       15.0
                  age 0.0043  0.002282        21.0        9.0
      lab_51301_daily 0.0026 -0.002013        24.0       12.0
          ccsr_BLD003 0.0017  0.000447        29.0       27.0

               feature   delta      shap  delta_rank  shap_rank
       lab_50931_daily -0.0445 -0.000206         1.0       40.0
       lab_51250_daily -0.0417 -0.011202         2.0        2.0
         num_diagnoses -0.0174  0.001373         6.0       16.0
       lab_50971_daily -0.0156  0.002172         7.0       11